# Qualitative Response Auto-Coding

Upload `flow_overview.csv`, `stimulus_key.csv`, and `qualitative_codebook.csv`, then run the cells in order. The notebook creates codebook-based theme counts for best/worst text explanations.

In [ ]:
import os, re, json, shutil
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content')
OUTPUT_DIR = DATA_DIR / 'qualitative_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_LABELS = {
    'random_noise': 'Random Noise',
    'random_vae_vector': 'Random-Plan VAE',
    'hapticgen': 'HapticGen',
    'llm_direct': 'Direct LLM Pattern',
    'semantic_vae': 'Semantic VAE',
}


## Upload CSV Files

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


## Load Data

In [ ]:
flow_overview = pd.read_csv(DATA_DIR / 'flow_overview.csv')
stimulus_key = pd.read_csv(DATA_DIR / 'stimulus_key.csv')
codebook = pd.read_csv(DATA_DIR / 'qualitative_codebook.csv')

stimulus_key['anonymous_id'] = stimulus_key['anonymous_id'].astype(str)
stimulus_key['method_label'] = stimulus_key['method'].map(METHOD_LABELS).fillna(stimulus_key['method'])

display(flow_overview.head())
display(stimulus_key.head())
display(codebook[['code_id', 'theme']])


## Extract Best/Worst Responses and Map IDs to Methods

In [ ]:
def split_ids(value):
    if pd.isna(value) or str(value).strip() == '':
        return []
    return [item.strip() for item in str(value).split('|') if item.strip()]

reason_groups = []
for _, row in flow_overview.iterrows():
    for selection_type, ids_col, reason_col in [('best', 'top3_ids', 'top_reason'), ('worst', 'bottom_ids', 'bottom_reason')]:
        ids = split_ids(row.get(ids_col))
        text = '' if pd.isna(row.get(reason_col)) else str(row.get(reason_col)).strip()
        if ids or text:
            reason_groups.append({
                'participant_id': row['participant_id'],
                'flow': row['flow'],
                'selection_type': selection_type,
                'selected_ids': '|'.join(ids),
                'reason_text': text,
            })

reason_groups = pd.DataFrame(reason_groups)
reason_groups.to_csv(OUTPUT_DIR / 'qualitative_reason_groups_extracted.csv', index=False)

rows = []
for _, row in reason_groups.iterrows():
    for anon_id in split_ids(row['selected_ids']):
        match = stimulus_key[(stimulus_key['anonymous_id'] == str(anon_id)) & (stimulus_key['flow'] == row['flow'])]
        if match.empty:
            method = method_label = variant = ''
        else:
            key = match.iloc[0]
            method, method_label, variant = key['method'], key['method_label'], key['variant']
        rows.append({**row.to_dict(), 'anonymous_id': str(anon_id), 'method': method, 'method_label': method_label, 'variant': variant})

reason_by_stimulus = pd.DataFrame(rows)
reason_by_stimulus.to_csv(OUTPUT_DIR / 'qualitative_reason_by_selected_stimulus.csv', index=False)
display(reason_by_stimulus.head())


## Keyword-Assisted Codebook Coding

In [ ]:
def normalize_text(text):
    text = '' if pd.isna(text) else str(text).lower()
    return re.sub(r'\s+', ' ', text).strip()

def parse_keywords(value):
    if pd.isna(value):
        return []
    return [kw.strip().lower() for kw in str(value).split(';') if kw.strip()]

keyword_map = {row['code_id']: parse_keywords(row['keyword_hints']) for _, row in codebook.iterrows()}

def keyword_code(text):
    normalized = normalize_text(text)
    codes, matched_terms = [], {}
    for code_id, keywords in keyword_map.items():
        hits = [kw for kw in keywords if kw in normalized]
        if hits:
            codes.append(code_id)
            matched_terms[code_id] = '|'.join(sorted(set(hits)))
    return (codes or ['uncoded']), matched_terms

coded_rows = []
for _, row in reason_by_stimulus.iterrows():
    codes, matched = keyword_code(row['reason_text'])
    for code_id in codes:
        coded_rows.append({**row.to_dict(), 'code_id': code_id, 'coding_source': 'keyword', 'matched_terms': matched.get(code_id, '')})

keyword_coded = pd.DataFrame(coded_rows)
keyword_coded.to_csv(OUTPUT_DIR / 'qualitative_coded_keyword.csv', index=False)
display(keyword_coded.head())


## Summarize Theme Counts

In [ ]:
theme_counts = (
    keyword_coded.groupby(['selection_type', 'flow', 'method_label', 'code_id'])
    .size().reset_index(name='count')
    .sort_values(['selection_type', 'flow', 'method_label', 'count'], ascending=[True, True, True, False])
)
overall_counts = (
    keyword_coded.groupby(['selection_type', 'code_id'])
    .size().reset_index(name='count')
    .sort_values(['selection_type', 'count'], ascending=[True, False])
)

theme_counts.to_csv(OUTPUT_DIR / 'theme_counts_by_method_flow_keyword.csv', index=False)
overall_counts.to_csv(OUTPUT_DIR / 'theme_counts_overall_keyword.csv', index=False)

display(theme_counts.head(30))
display(overall_counts)


## Optional LLM-Assisted Coding

In [ ]:
# Optional. Run this section only if you want model-assisted coding.
# !pip install -q openai
# os.environ['OPENAI_API_KEY'] = 'your_api_key'

USE_LLM = False
OPENAI_MODEL = 'gpt-4.1-mini'

def make_codebook_prompt(codebook_df):
    lines = []
    for _, row in codebook_df.iterrows():
        lines.append(f"- {row['code_id']}: {row['theme']}. Definition: {row['definition']} Include when: {row['include_when']} Exclude when: {row['exclude_when']}")
    return '\n'.join(lines)

def llm_code_responses(reason_df, codebook_df):
    from openai import OpenAI
    client = OpenAI()
    allowed_codes = codebook_df['code_id'].tolist() + ['uncoded']
    codebook_text = make_codebook_prompt(codebook_df)
    schema = {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'codes': {'type': 'array', 'items': {'type': 'string', 'enum': allowed_codes}},
            'confidence': {'type': 'number'},
            'rationale': {'type': 'string'},
        },
        'required': ['codes', 'confidence', 'rationale'],
    }
    outputs = []
    for idx, row in reason_df.iterrows():
        prompt = f'''Assign one or more codebook codes to this haptic-study response. Use uncoded only if no code fits.\n\nCodebook:\n{codebook_text}\n\nContext: flow={row['flow']}; selection={row['selection_type']}; method={row['method_label']}\nResponse: {row['reason_text']}'''
        response = client.responses.create(
            model=OPENAI_MODEL,
            input=[{'role': 'system', 'content': 'You are a careful qualitative research coding assistant.'}, {'role': 'user', 'content': prompt}],
            text={'format': {'type': 'json_schema', 'name': 'qualitative_codes', 'strict': True, 'schema': schema}},
        )
        parsed = json.loads(response.output_text)
        for code_id in parsed.get('codes', []) or ['uncoded']:
            outputs.append({**row.to_dict(), 'code_id': code_id, 'coding_source': 'llm', 'confidence': parsed.get('confidence', ''), 'rationale': parsed.get('rationale', '')})
    return pd.DataFrame(outputs)

if USE_LLM:
    llm_coded = llm_code_responses(reason_by_stimulus, codebook)
    llm_coded.to_csv(OUTPUT_DIR / 'qualitative_coded_llm.csv', index=False)
    display(llm_coded.head())
else:
    print('USE_LLM is False. Keyword-coded outputs are ready.')


## Download Outputs

In [ ]:
from google.colab import files
zip_path = shutil.make_archive('/content/qualitative_outputs', 'zip', OUTPUT_DIR)
files.download(zip_path)
